# Claims in reviews
Controllo quali recensioni menzionano le feature Bosch.

In [ ]:
import re
import sqlite3
import pandas as pd

DB_PATH = "bosch_dw.db"   # cambia il percorso se necessario
MIN_RATING = 4

## Definizione dei claim
Ogni claim ha una lista di pattern regex. Un match su titolo+testo conta come menzione.

In [ ]:
CLAIMS = {
    "home_connect": [
        r"home\s*connect", r"\bapp\b", r"wi-?fi", r"smartphone",
        r"remote(ly)?", r"connected?\s+appliance",
    ],
    "smart_start": [
        r"smart\s*start", r"off.?peak", r"energy\s+tari",
        r"cheap(er)?\s+(rate|energy|electricity)", r"electricity\s+price",
    ],
    "programme_download": [
        r"programme\s+download", r"download.*program",
        r"new\s+program", r"update.*program",
    ],
    "intelligent_prog": [
        r"intelligent\s+programme", r"auto.?sens", r"sensor.*wash",
        r"auto\s+program", r"automatic(ally)?\s+(detect|select|adjust)",
    ],
    "wash_assistant": [
        r"wash\s+assistant", r"washing\s+assistant",
    ],
    "efficient_dry": [
        r"efficient\s*dry", r"dry(ing)?\s+result",
        r"dishes?\s+(are\s+|came?\s+out\s+)?dry", r"condensation\s+dry",
        r"automatic\s+door(\s+opening)?", r"door\s+(opening|open(s)?)",
        r"auto[\s-]?open",
        r"(door\s+)?opens?\s+(automatically|on\s+its\s+own|by\s+itself)",
        r"self[\s-]?opening(\s+door)?",
    ],
    "perfect_dry": [
        r"perfect\s*dry", r"plastic(s)?\s+(are\s+|came?\s+out\s+)?dry",
        r"dry(ing)?(\s+on)?\s+plastic", r"zeolite",
    ],
    "extra_clean_zone": [
        r"extra\s+clean\s+zone", r"intensive\s+zone", r"pots?\s+and\s+pans?",
        r"heavily\s+soil", r"intensiv",
    ],
    "hygiene_certified": [
        r"hygie(ne|nic)", r"bacteria", r"germ", r"sanitiz",
        r"disinfect", r"70.?°?.?c", r"certified", r"allergen",
    ],
    "remote_diagnostics": [
        r"remote\s+diagnos", r"error\s+(code|message)",
        r"technician", r"service\s+call", r"fault\s+detect",
    ],
    "green_collection": [
        r"green\s+collection", r"eco.?friendly", r"sustainab",
        r"recycled?\s+steel", r"environmental", r"carbon",
    ],
    "silent_power_drive": [
        r"silent\s*power\s*drive", r"\bsilent\b", r"\bquiet\b",
        r"\bnoise\b", r"\bnoisy\b", r"\bdB\b", r"decibel", r"sound\s+level",
    ],
    "aquastop": [
        r"aqua\s*stop", r"flood(ing)?", r"leak(age|s|ed|ing)?",
        r"water\s+(protection|damage|leak)", r"anti.?flood",
    ],
    "time_light": [
        r"time\s+light", r"remaining\s+time", r"countdown",
        r"floor\s+(light|project)", r"projected?\s+(on\s+)?floor", r"info\s*light",
    ],
    "emotion_light": [
        r"emotion\s+light", r"interior\s+light", r"inside\s+light",
        r"illuminat", r"light\s+inside",
    ],
    "open_assist": [
        r"open\s+assist", r"auto.?open", r"door\s+(open|pop)",
        r"opens?\s+automatically", r"automatic\s+door",
    ],
    "rackmatic": [
        r"rackmatic", r"adjustable\s+(rack|basket|height)",
        r"rack\s+height", r"height\s+adjust", r"upper\s+basket\s+adjust",
    ],
    "max_flex_pro": [
        r"max\s+flex\s+pro", r"flex(ible)?\s+(rack|basket|tine)",
        r"fold(able|ing)?\s+(tine|spike|prong)", r"large\s+(pot|pan|item)",
    ],
    "dosage_assist": [
        r"dosage\s+assist", r"detergent\s+dissolv", r"tablet\s+dissolv",
        r"dissolv.*detergent", r"detergent\s+compart",
    ],
    # ── due claim aggiunti ──────────────────────────────────────────────────
    "vario_hinge": [
        r"vario\s*hinge", r"\bhinge\b", r"door\s+hinge",
        r"fold(ing)?\s+door", r"door\s+fold",
    ],
    "favorite_program": [
        r"favou?rite\s+program", r"saved\s+program",
        r"my\s+program", r"custom\s+program", r"personal\s+program",
        r"preferred\s+program",
    ],
}

# Pattern per il nome esatto del claim (match diretto sul brand name)
CLAIM_EXACT_NAME = {
    "home_connect":       r"home\s*connect",
    "smart_start":        r"smart\s*start",
    "programme_download": r"programme\s*download",
    "intelligent_prog":   r"intelligent\s*programme",
    "wash_assistant":     r"wash\s*assistant",
    "efficient_dry":      r"efficient\s*dry",
    "perfect_dry":        r"perfect\s*dry",
    "extra_clean_zone":   r"extra\s*clean\s*zone",
    "hygiene_certified":  r"hygie\w+\s*certif\w+",
    "remote_diagnostics": r"remote\s*diagnos\w+",
    "green_collection":   r"green\s*collection",
    "silent_power_drive": r"silent\s*power\s*drive",
    "aquastop":           r"aqua\s*stop",
    "time_light":         r"time\s*light",
    "emotion_light":      r"emotion\s*light",
    "open_assist":        r"open\s*assist",
    "rackmatic":          r"rackmatic",
    "max_flex_pro":       r"max\s*flex\s*pro",
    "dosage_assist":      r"dosage\s*assist",
    "vario_hinge":        r"vario\s*hinge",
    "favorite_program":   r"favou?rite\s*program",
}

# Compile una volta sola
COMPILED = {
    claim: [re.compile(kw, re.IGNORECASE) for kw in patterns]
    for claim, patterns in CLAIMS.items()
}
COMPILED_EXACT = {
    claim: re.compile(pattern, re.IGNORECASE)
    for claim, pattern in CLAIM_EXACT_NAME.items()
}

print(f"{len(CLAIMS)} claim definiti")

## Caricamento dati dal DB

In [ ]:
con = sqlite3.connect(DB_PATH)

products = pd.read_sql("SELECT * FROM dim_product", con)
reviews  = pd.read_sql("SELECT * FROM fact_review",  con)

con.close()

print(f"Prodotti: {len(products)}")
print(f"Recensioni: {len(reviews)}")

## Matching
Per ogni recensione: concatena titolo+testo e testa i pattern di ciascun claim.

In [ ]:
def matches_any(text: str, patterns: list) -> bool:
    return bool(text) and any(p.search(text) for p in patterns)

text_col = (reviews["title_en"].fillna("") + " " + reviews["text_en"].fillna(""))

for claim, patterns in COMPILED.items():
    reviews[f"mentions_{claim}"] = text_col.apply(lambda t: matches_any(t, patterns))

for claim, pattern in COMPILED_EXACT.items():
    reviews[f"exact_{claim}"] = text_col.apply(lambda t: bool(pattern.search(t)))

print("Colonne mentions_* aggiunte:", reviews.filter(like="mentions_").shape[1])
print("Colonne exact_*    aggiunte:", reviews.filter(like="exact_").shape[1])

## Join con flag di prodotto
Aggiunge `has_<claim>` dalla dim_product per sapere se il prodotto *ha* effettivamente quella feature.

In [ ]:
has_cols = [c for c in products.columns if c.startswith("has_")]

reviews = reviews.merge(
    products[["product_id"] + has_cols],
    on="product_id",
    how="left",
)

print("Colonne has_* aggiunte:", len(has_cols))
reviews[["product_id", "rating"] + has_cols].head(3)

## Statistiche per claim
Solo recensioni con rating ≥ MIN_RATING. Per ogni claim: quante recensioni provengono da prodotti che **hanno** quella feature, e quante la menzionano.

In [ ]:
filtered = reviews[reviews["rating"] >= MIN_RATING].copy()

rows = []
for claim in CLAIMS:
    has_col     = f"has_{claim}"
    mention_col = f"mentions_{claim}"

    if has_col not in filtered.columns:
        # il flag non esiste in dim_product per questo claim
        rows.append({"claim": claim, "eligible": None, "mentions": None, "mention_%": None})
        continue

    eligible = filtered[filtered[has_col] == 1]
    n        = len(eligible)
    mentions = eligible[mention_col].sum()
    exact    = eligible[f"exact_{claim}"].sum() if f"exact_{claim}" in eligible.columns else 0

    rows.append({
        "claim":     claim,
        "eligible":  n,
        "mentions":  int(mentions),
        "mention_%": round(100 * mentions / n, 1) if n else 0,
        "exact":     int(exact),
        "exact_%":   round(100 * exact / n, 1) if n else 0,
    })

stats = pd.DataFrame(rows).sort_values("mention_%", ascending=False)
stats

## Ispezione manuale
Modifica `CLAIM_TO_INSPECT` per vedere le recensioni che menzionano un claim specifico.

In [ ]:
CLAIM_TO_INSPECT = "efficient_dry"   # <-- cambia qui

EXACT_ONLY = False   # True = solo recensioni con nome esatto del claim

match_col = f"exact_{CLAIM_TO_INSPECT}" if EXACT_ONLY else f"mentions_{CLAIM_TO_INSPECT}"

mask = (
    (reviews["rating"] >= MIN_RATING) &
    (reviews[f"has_{CLAIM_TO_INSPECT}"] == 1) &
    (reviews[match_col] == True)
)

sample = reviews[mask][["product_id", "country", "rating", "title_en", "text_en"]].head(10)
pd.set_option("display.max_colwidth", 120)
sample